In [1]:
import numpy as np
import itertools
from collections import defaultdict

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver, SamplingVQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit.circuit.library import TwoLocal

# PROBLEM SETUP
dist = np.array([
    [0, 3.9, 6, 5],
    [4, 0, 3, 2],
    [6, 3, 0, 4],
    [5, 2, 4, 0],
], dtype=float)

scenic = {1: 5.0, 2: 9.0, 3: 6.0} 
pois = [1, 2, 3]
slots = [1, 2]
A, B = 12.0, 10.0

# BUILD QUBO
qp = QuadraticProgram("toy_hiking_route")
for i in pois:
    for p in slots:
        qp.binary_var(name=f"x_{i}_{p}")

linear, quadratic = defaultdict(float), defaultdict(float)
constant = 0.0

for i in pois:
    linear[f"x_{i}_1"] += dist[0][i] - scenic[i]
    linear[f"x_{i}_2"] += dist[i][0] - scenic[i]

for i in pois:
    for j in pois:
        quadratic[(f"x_{i}_1", f"x_{j}_2")] += dist[i][j]

for p in slots:
    constant += A
    for i in pois:
        linear[f"x_{i}_{p}"] += -A
    for i, j in itertools.combinations(pois, 2):
        quadratic[(f"x_{i}_{p}", f"x_{j}_{p}")] += 2 * A

for i in pois:
    quadratic[(f"x_{i}_1", f"x_{i}_2")] += B

qp.minimize(constant=constant, linear=dict(linear), quadratic=dict(quadratic))
op, offset = qp.to_ising()

def decode_solution(xvec):
    var_names = [f"x_{i}_{p}" for i in pois for p in slots]
    sol = {name: int(round(val)) for name, val in zip(var_names, xvec)}
    first, second = None, None
    for i in pois:
        if sol[f"x_{i}_1"] == 1: first = i
        if sol[f"x_{i}_2"] == 1: second = i
    return [0, first, second, 0]

sampler = StatevectorSampler()

# Exact
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)

# QAOA
qaoa = MinimumEigenOptimizer(QAOA(sampler=sampler, optimizer=COBYLA(maxiter=300), reps=2)).solve(qp)

# VQE - 
ansatz = TwoLocal(num_qubits=op.num_qubits, rotation_blocks=["ry", "rz"], entanglement_blocks="cz", reps=3)
vqe = MinimumEigenOptimizer(SamplingVQE(sampler=sampler, ansatz=ansatz, optimizer=COBYLA(maxiter=800))).solve(qp)

# 4. OUTPUT
print(f"{'Solver':<10} | {'Route':<15} | {'Cost':<10}")
print("-" * 40)
print(f"{'EXACT':<10} | {str(decode_solution(exact.x)):<15} | {exact.fval:<10.2f}")
print(f"{'QAOA':<10} | {str(decode_solution(qaoa.x)):<15} | {qaoa.fval:<10.2f}")
print(f"{'VQE':<10} | {str(decode_solution(vqe.x)):<15} | {vqe.fval:<10.2f}")

/local/sugale/quantum_error_mitigation/benasquehack/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/local/sugale/quantum_error_mitigation/benasquehack/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/local/sugale/quantum_error_mitigation/benasquehack/lib/python3.12/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])
/tmp/ipykernel_282911/277296597.py:73: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(num_qubits=op.num

Solver     | Route           | Cost      
----------------------------------------
EXACT      | [0, 1, 2, 0]    | -1.10     
QAOA       | [0, 1, 2, 0]    | -1.10     
VQE        | [0, 1, 2, 0]    | -1.10     
